# Lezione 3 — Regressione e classificazione con Keras

## Obiettivi di apprendimento
- Costruire il tuo primo modello di deep learning usando la libreria Keras.
- Addestrare una rete neurale di **regressione** che predice la resistenza del
  calcestruzzo a partire dai suoi ingredienti.
- Addestrare una rete neurale di **classificazione** che riconosce cifre
  scritte a mano (MNIST) usando un'architettura densa.
- Capire perché lo strato di output e la funzione di perdita cambiano tra
  un problema di regressione e uno di classificazione.
- Confrontare il flusso di lavoro di Keras (`Sequential` → `compile` → `fit`)
  con il backpropagation manuale che hai programmato nella lezione precedente.

## Cosa costruiremo
1. Un modello di regressione con due strati nascosti di 50 nodi che predice la
   resistenza del calcestruzzo (`concrete_data.csv`, dataset pubblico di IBM).
2. Un modello di classificazione con una rete densa che riconosce cifre da 0
   a 9 sul dataset MNIST, incluso direttamente in Keras.

Tutto il codice di questa lezione gira su Google Colab senza bisogno di
credenziali né token: il dataset del calcestruzzo si scarica tramite URL pubblico
e MNIST è incluso in `keras.datasets`.

Keras e TensorFlow sono già preinstallati su Colab, quindi non serve
nessun `!pip install` per questa lezione.

In [ ]:
# Verifichiamo la versione di Keras/TensorFlow disponibile nell'ambiente.
# (Su Colab sono già preinstallati; se esegui questo localmente e fallisce,
# installa con: pip install tensorflow)
import keras
print("Keras:", keras.__version__)

## Parte 1 — Regressione: resistenza del calcestruzzo

**Promemoria teorico (capitolo 5 del manuale):** in un problema di
**regressione** la rete predice un numero continuo. Il flusso con Keras è
sempre lo stesso: caricare e ripulire i dati → separare predittori e
target → normalizzare (z-score) → definire il modello con `Sequential` →
compilare → `fit`.

Il dataset (`concrete_data.csv`) descrive quasi 1000 campioni di calcestruzzo:
cemento, loppa d'altoforno, cenere volante, acqua, superplastificante,
aggregato grosso, aggregato fine ed età della miscela. La colonna target è
`Strength`, la resistenza a compressione in MPa.

In [ ]:
import pandas as pd
import numpy as np

# Carichiamo il dataset direttamente dal bucket S3 pubblico di IBM.
# È lo stesso URL usato nel materiale originale del corso; non richiede
# credenziali né token di alcun tipo.
filepath = 'https://s3-api.us-geo.objectstorage.softlayer.net/cf-courses-data/CognitiveClass/DL0101EN/labs/data/concrete_data.csv'
concrete_data = pd.read_csv(filepath)

print(concrete_data.shape)
concrete_data.head()

In [ ]:
# Controlliamo che non ci siano valori nulli: se il dataset fosse sporco,
# questo sarebbe il momento di ripulirlo prima di proseguire.
concrete_data.isnull().sum()

In [ ]:
# Separiamo i predittori (tutte le colonne tranne "Strength") dal target.
concrete_data_columns = concrete_data.columns
predictors = concrete_data[concrete_data_columns[concrete_data_columns != 'Strength']]
target = concrete_data['Strength']

# Normalizzazione z-score: (x - media) / deviazione standard.
predictors_norm = (predictors - predictors.mean()) / predictors.std()

n_cols = predictors_norm.shape[1]  # numero di colonne di input
print("Numero di predittori:", n_cols)
predictors_norm.head()

**Architettura del modello di regressione** (identica a quella del materiale
originale di IBM): strato di input con `n_cols` valori, due strati nascosti
`Dense` di 50 nodi con attivazione ReLU, e uno strato di output di un solo
nodo **senza attivazione** (perché vogliamo un numero reale senza restrizioni
di intervallo). Compiliamo con l'ottimizzatore `adam` e perdita
`mean_squared_error`, la combinazione standard per la regressione.

In [ ]:
from keras.models import Sequential
from keras.layers import Dense, Input

def regression_model():
    model = Sequential()
    model.add(Input(shape=(n_cols,)))
    model.add(Dense(50, activation='relu'))
    model.add(Dense(50, activation='relu'))
    model.add(Dense(1))  # output: un numero, senza attivazione

    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

model = regression_model()
model.summary()

In [ ]:
# Addestriamo e validiamo allo stesso tempo con fit(): 30% dei dati per
# la validazione, 100 epoche. Su Colab questo richiede solo pochi secondi perché il
# dataset è piccolo (~1000 righe).
history = model.fit(predictors_norm, target, validation_split=0.3, epochs=100, verbose=2)

In [ ]:
# Confrontiamo l'errore finale di addestramento con quello di validazione.
print("loss finale (addestramento):", history.history['loss'][-1])
print("val_loss finale (validazione):", history.history['val_loss'][-1])

## Parte 2 — Classificazione: cifre scritte a mano (MNIST)

**Promemoria teorico (capitolo 6 del manuale):** nella classificazione lo strato
di output necessita di un nodo per classe con attivazione **softmax**, che
converte gli output in una distribuzione di probabilità. Le etichette
si preparano con **one-hot encoding**, e la perdita diventa
`categorical_crossentropy`. Poiché usiamo una rete densa (non ancora
convoluzionale, quella è la prossima lezione), prima bisogna **appiattire** ogni immagine
28x28 in un vettore di 784 valori.

In [ ]:
from keras.datasets import mnist
from keras.utils import to_categorical
import matplotlib.pyplot as plt

# MNIST è incluso in Keras: non richiede download manuale né credenziali.
(X_train, y_train), (X_test, y_test) = mnist.load_data()
print("Addestramento:", X_train.shape)
print("Test:", X_test.shape)

# Visualizziamo la prima immagine del set di addestramento.
plt.imshow(X_train[0], cmap='gray')
plt.title(f"Etichetta reale: {y_train[0]}")
plt.show()

In [ ]:
# Appiattiamo ogni immagine 28x28 in un vettore di 784 valori (28*28), perché
# una rete Dense si aspetta vettori, non matrici.
num_pixels = X_train.shape[1] * X_train.shape[2]
X_train = X_train.reshape(X_train.shape[0], num_pixels).astype('float32')
X_test = X_test.reshape(X_test.shape[0], num_pixels).astype('float32')

# Normalizziamo i pixel da [0, 255] a [0, 1].
X_train = X_train / 255
X_test = X_test / 255

# One-hot encoding delle etichette: ogni cifra diventa un vettore di
# 10 posizioni con un unico 1 nella posizione corretta.
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)
num_classes = y_test.shape[1]
print("Numero di classi:", num_classes)

In [ ]:
def classification_model():
    model = Sequential()
    model.add(Input(shape=(num_pixels,)))
    model.add(Dense(num_pixels, activation='relu'))
    model.add(Dense(100, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))  # una probabilità per cifra

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

clf_model = classification_model()
clf_model.summary()

In [ ]:
# Addestriamo usando l'insieme di test come validazione a ogni epoca.
# NOTA: con 10 epoche e il dataset completo, questo può richiedere diversi
# minuti su un Colab con CPU. Se ti servono risultati più rapidi per
# proiettare in lezione, riduci epochs a 3.
clf_history = clf_model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=10, verbose=2)

scores = clf_model.evaluate(X_test, y_test, verbose=0)
print('Precisione: {:.4f} \n Errore: {:.4f}'.format(scores[1], 1 - scores[1]))

### Salvare e ricaricare il modello

Keras permette di salvare un modello addestrato in un unico file `.keras` e
recuperarlo in seguito senza doverlo riaddestrare da zero.

In [ ]:
clf_model.save('classification_model.keras')

pretrained_model = keras.saving.load_model('classification_model.keras')
print("Modello ricaricato correttamente.")

## Riepilogo della lezione

- Con Keras, qualsiasi modello di deep learning segue lo schema: preparare
  i dati → `Sequential` → `compile` → `fit`.
- **Regressione**: output di 1 nodo senza attivazione, perdita
  `mean_squared_error`.
- **Classificazione**: output con un nodo per classe e attivazione `softmax`,
  etichette in one-hot, perdita `categorical_crossentropy`.
- Una rete densa richiede che le immagini vengano appiattite in vettori prima di
  poterle elaborare; nella prossima lezione vedrai come le reti
  convoluzionali evitano questo passaggio e sfruttano la struttura spaziale
  dell'immagine.

**Prosegui con:**
- Capitoli del manuale: [`05_regresion_keras.md`](../manual/05_regresion_keras_IT.md) e [`06_clasificacion_keras.md`](../manual/06_clasificacion_keras_IT.md)
- Notebook di pratica di questa lezione: [`clase03_practica_regresion_clasificacion_keras.ipynb`](../notebooks_practica/clase03_practica_regresion_clasificacion_keras_IT.ipynb)